# Experiment: CytoRAG Qwen API Embedding and Chunking Study

Objective:
- Вынести эксперименты из `embedding_models_study.ipynb` в отдельный ноутбук.
- Убрать локальный `vllm` и `LocalLLM`: для генерации и LLM-оценки использовать только Qwen API по схеме из `models.ipynb`.
- Сохранить два сценария исследования: сравнение embedding-моделей и сравнение chunking-стратегий.

Важно:
- Эмбеддинги и reranker остаются локальными, потому что это предмет самого исследования.
- Qwen используется только как внешняя LLM для генерации ответов и judge/RAGAS-оценки.


In [ ]:
%pip install -q -U "transformers>=4.55.2,<5" "tokenizers>=0.21.1,<0.22" "sentence-transformers>=3.0,<4" "faiss-cpu>=1.8,<2" "rank-bm25>=0.2.2" "python-dotenv>=1.0" "requests==2.32.4" "datasets>=4,<5" "ragas>=0.4,<0.5" "langchain-core" "langchain-huggingface>=0.3.1" "pandas>=2,<3"


In [1]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import faiss
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
#from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
load_dotenv()

def resolve_base_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "bethesda_ground_truth.json").exists():
        return cwd
    candidate = cwd / "CytoRAG"
    if (candidate / "bethesda_ground_truth.json").exists():
        return candidate
    raise FileNotFoundError("Не удалось найти bethesda_ground_truth.json ни в текущей директории, ни в ./CytoRAG")

BASE_DIR = resolve_base_dir()
DATA_PATH = BASE_DIR / "bethesda_ground_truth.json"
EXISTING_RAGAS_CSV = BASE_DIR / "ragas_evaluation_results.csv"
ARTIFACT_DIR = BASE_DIR / "artifacts" / "qwen_api_embedding_chunking_study"
EMBEDDING_ARTIFACT_DIR = ARTIFACT_DIR / "embedding_study"
CHUNKING_ARTIFACT_DIR = ARTIFACT_DIR / "chunking_study"
EMBEDDING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_DEVICE = "cpu"
RERANKER_NAME = "DiTy/cross-encoder-russian-msmarco"
DENSE_TOP_K = 5
FINAL_TOP_K = 3

EXPERIMENT_MODE = "chunking_only"  # one of: "both", "embedding_only", "chunking_only"
RUN_LLM_MANUAL_METRICS = False
RUN_RAGAS = False

if EXPERIMENT_MODE not in {"both", "embedding_only", "chunking_only"}:
    raise ValueError("EXPERIMENT_MODE must be one of: both, embedding_only, chunking_only")

RUN_RETRIEVAL_SWEEP = EXPERIMENT_MODE in {"both", "embedding_only"}
RUN_CHUNKING_SWEEP = EXPERIMENT_MODE in {"both", "chunking_only"}
HAS_HF_TOKEN = bool(os.getenv("HUGGINGFACE_API_KEY"))

QWEN_MODEL_REPO = os.getenv("QWEN_MODEL_REPO", "Qwen/Qwen2.5-7B-Instruct")
QWEN_TEMPERATURE = float(os.getenv("QWEN_TEMPERATURE", "0.1"))
QWEN_MAX_NEW_TOKENS = int(os.getenv("QWEN_MAX_NEW_TOKENS", "512"))

print({
    "base_dir": str(BASE_DIR),
    "artifacts": str(ARTIFACT_DIR),
    "embedding_device": EMBEDDING_DEVICE,
    "reranker_device": RERANKER_DEVICE,
    "experiment_mode": EXPERIMENT_MODE,
    "qwen_model_repo": QWEN_MODEL_REPO,
    "has_hf_token": HAS_HF_TOKEN,
    "run_retrieval_sweep": RUN_RETRIEVAL_SWEEP,
    "run_llm_manual_metrics": RUN_LLM_MANUAL_METRICS,
    "run_ragas": RUN_RAGAS,
    "run_chunking_sweep": RUN_CHUNKING_SWEEP,
})


d:\Mephi\22kaf\6sem\УИР\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'base_dir': 'd:\\Mephi\\22kaf\\6sem\\УИР\\CytoRAG', 'artifacts': 'd:\\Mephi\\22kaf\\6sem\\УИР\\CytoRAG\\artifacts\\qwen_api_embedding_chunking_study', 'embedding_device': 'cpu', 'reranker_device': 'cpu', 'experiment_mode': 'chunking_only', 'qwen_model_repo': 'Qwen/Qwen2.5-7B-Instruct', 'has_hf_token': True, 'run_retrieval_sweep': False, 'run_llm_manual_metrics': False, 'run_ragas': False, 'run_chunking_sweep': True}


## Plan

- Hypothesis 1: среди embedding-моделей будут заметные различия по `hit_rate` и `mrr` на Bethesda-кейсах.
- Hypothesis 2: short structural chunking сможет улучшить качество retrieval даже при фиксированном embedding-backbone.
- Chunking defaults to the lightest embedding backbone (`rubert_tiny2_baseline`), чтобы не ждать долгую загрузку `bge-m3` на CPU.
- Variables to sweep: `MODEL_SPECS`, `CHUNKING_STRATEGIES`.
- Metrics to record: `hit_rate`, `mrr`, optional manual judge metrics, `ragas` metrics, количество индексируемых чанков.
- LLM policy: только `Qwen/Qwen2.5-7B-Instruct` через Hugging Face API.
- Launch presets: `EXPERIMENT_MODE="both"`, `"embedding_only"` или `"chunking_only"`.


In [2]:
@dataclass
class ModelSpec:
    label: str
    model_name: str
    batch_size: int
    max_length: int
    notes: str

MODEL_SPECS: List[ModelSpec] = [
    ModelSpec("rubert_tiny2_baseline", "cointegrated/rubert-tiny2", 32, 384, "Current lightweight baseline closest to test.ipynb"),
    ModelSpec("rubert_base_sentence", "DeepPavlov/rubert-base-cased-sentence", 16, 384, "Russian sentence encoder"),
    ModelSpec("bge_m3", "BAAI/bge-m3", 4, 512, "Multilingual retrieval model; no query prefix needed for bge-m3"),
    ModelSpec("biolord_2023_m", "FremyCompany/BioLORD-2023-M", 8, 512, "Multilingual biomedical encoder"),
    ModelSpec("pubmedbert_fulltext", "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext", 8, 384, "English biomedical control baseline"),
]

def clear_torch_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def tokenize_ru_medical(text: str) -> List[str]:
    return re.findall(r"[A-Za-zА-Яа-яЁё0-9№/-]+", text.lower())

def extract_case_anchor(text: str, max_len: int = 110) -> str:
    first_sentence = re.split(r"(?<=[.!?])\s+", text.strip())[0]
    return first_sentence[:max_len].rstrip(" ,;:")

def extract_bethesda_label(text: str) -> Optional[str]:
    match = re.search(r"bethesda\s*[-–]\s*([ivx]+)", text, flags=re.IGNORECASE)
    return match.group(1).upper() if match else None

def build_eval_rows(raw_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    eval_rows: List[Dict[str, Any]] = []
    for row in raw_rows:
        anchor = extract_case_anchor(row["simulated_context"])
        eval_rows.append({
            "id": int(row["id"]),
            "original_query": row["query"],
            "retrieval_query": f"Определи диагностическую категорию Bethesda для клинического случая: {anchor}",
            "case_anchor": anchor,
            "simulated_context": row["simulated_context"],
            "ground_truth": row["ground_truth"],
            "bethesda_label": extract_bethesda_label(row["ground_truth"]),
        })
    return eval_rows

def summarize_existing_ragas(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    summary = {"model": "existing_csv_baseline"}
    for metric in ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
        if metric in df.columns:
            series = pd.to_numeric(df[metric], errors="coerce")
            summary[metric] = round(series.mean() * 100, 2) if series.notna().any() else np.nan
    return pd.DataFrame([summary])

def dataframe_to_markdown(frame: pd.DataFrame, columns: Optional[List[str]] = None) -> str:
    if frame.empty:
        return "_No rows to display._"
    view = frame.copy()
    if columns is not None:
        view = view[columns]
    def normalize_value(value: Any) -> str:
        if value is None:
            return ""
        if isinstance(value, float):
            if np.isnan(value):
                return ""
            return f"{value:.2f}"
        return str(value)
    headers = list(view.columns)
    rows = [[normalize_value(value) for value in row] for row in view.to_numpy().tolist()]
    widths = [len(header) for header in headers]
    for row in rows:
        widths = [max(width, len(cell)) for width, cell in zip(widths, row)]
    def format_row(row: List[str]) -> str:
        return "| " + " | ".join(cell.ljust(width) for cell, width in zip(row, widths)) + " |"
    separator = "| " + " | ".join("-" * width for width in widths) + " |"
    lines = [format_row(headers), separator]
    lines.extend(format_row(row) for row in rows)
    return "\n".join(lines)

raw_rows = json.loads(DATA_PATH.read_text(encoding="utf-8"))
eval_rows = build_eval_rows(raw_rows)
documents = [(row["simulated_context"], {"case_id": row["id"], "case_anchor": row["case_anchor"]}) for row in eval_rows]
existing_ragas_baseline_df = summarize_existing_ragas(EXISTING_RAGAS_CSV)

preview_df = pd.DataFrame({
    "id": [row["id"] for row in eval_rows],
    "case_anchor": [row["case_anchor"] for row in eval_rows],
    "bethesda_label": [row["bethesda_label"] for row in eval_rows],
})

print(f"Loaded {len(eval_rows)} Bethesda cases from {DATA_PATH.name}")
display(preview_df.head())
if not existing_ragas_baseline_df.empty:
    display(existing_ragas_baseline_df)


Loaded 10 Bethesda cases from bethesda_ground_truth.json


,id,case_anchor,bethesda_label
0,1,П 29 (№3683/20).,I
1,2,Пр.,I
2,3,Л 11 (№1995/20).,NaN
3,4,П 19 (№1405/22).,II
4,5,П 7 (№2705/21).,II


,model,context_precision,context_recall,faithfulness,answer_relevancy
0,existing_csv_baseline,91.67,58.33,83.33,NaN


In [3]:
class QwenAPIPipeline:
    def __init__(
        self,
        repo_id: str = QWEN_MODEL_REPO,
        temperature: float = QWEN_TEMPERATURE,
        max_new_tokens: int = QWEN_MAX_NEW_TOKENS,
    ):
        self.repo_id = repo_id
        self.temperature = 0.1 if temperature < 0.1 else temperature
        self.max_new_tokens = max_new_tokens
        self.hf_token = os.getenv("HUGGINGFACE_API_KEY")
        if not self.hf_token:
            raise ValueError("HUGGINGFACE_API_KEY не найден в .env. Qwen API недоступен.")
        self._model: Optional[ChatHuggingFace] = None

    def get_chat_model(self) -> ChatHuggingFace:
        if self._model is None:
            endpoint = HuggingFaceEndpoint(
                repo_id=self.repo_id,
                task="text-generation",
                huggingfacehub_api_token=self.hf_token,
                temperature=self.temperature,
                max_new_tokens=self.max_new_tokens,
            )
            self._model = ChatHuggingFace(llm=endpoint)
        return self._model

    def get_response(self, prompt: str, system_prompt: str = "You are a helpful assistant.") -> str:
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt),
        ]
        response = self.get_chat_model().invoke(messages)
        return str(response.content).strip()

_qwen_api: Optional[QwenAPIPipeline] = None

def get_qwen_api() -> QwenAPIPipeline:
    global _qwen_api
    if _qwen_api is None:
        _qwen_api = QwenAPIPipeline()
    return _qwen_api

print({"qwen_model_repo": QWEN_MODEL_REPO, "temperature": QWEN_TEMPERATURE, "client_initialized": False})


{'qwen_model_repo': 'Qwen/Qwen2.5-7B-Instruct', 'temperature': 0.1, 'client_initialized': False}


In [4]:
class AdvancedRAGPipeline:
    def __init__(
        self,
        spec: ModelSpec,
        reranker_name: str = RERANKER_NAME,
        embedding_device: str = EMBEDDING_DEVICE,
        reranker_device: str = RERANKER_DEVICE,
    ):
        self.spec = spec
        self.embedding_device = embedding_device
        self.reranker_device = reranker_device
        self.reranker_name = reranker_name

        print(f"Loading embedding model: {spec.model_name} on {embedding_device}")
        self.embedding_model = SentenceTransformer(spec.model_name, device=embedding_device)
        self.embedding_model.max_seq_length = min(spec.max_length, self.embedding_model.max_seq_length)

        print(f"Loading reranker: {reranker_name} on {reranker_device}")
        self.cross_encoder = CrossEncoder(reranker_name, device=reranker_device)

        self.chunks: List[Dict[str, Any]] = []
        self.bm25: Optional[BM25Okapi] = None
        self.index: Optional[faiss.IndexFlatIP] = None

    def process_documents(self, documents: List[Tuple[str, Dict[str, Any]]]) -> None:
        self.chunks = [{"text": text, "metadata": metadata} for text, metadata in documents]
        tokenized_corpus = [tokenize_ru_medical(chunk["text"]) for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

        embeddings = self.embedding_model.encode(
            [chunk["text"] for chunk in self.chunks],
            batch_size=self.spec.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)
        print(f"Indexed {len(self.chunks)} documents")

    def search(self, query: str, dense_top_k: int = DENSE_TOP_K, final_top_k: int = FINAL_TOP_K) -> List[Dict[str, Any]]:
        if self.bm25 is None or self.index is None:
            raise RuntimeError("Documents are not indexed. Call process_documents() first.")

        tokenized_query = tokenize_ru_medical(query)
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_idx = np.argsort(bm25_scores)[::-1][:dense_top_k]

        query_embedding = self.embedding_model.encode(
            [query],
            batch_size=1,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        _, dense_idx = self.index.search(query_embedding, dense_top_k)

        candidate_indices = [idx for idx in dict.fromkeys(list(bm25_idx) + list(dense_idx[0])) if idx != -1]
        if not candidate_indices:
            return []

        candidate_chunks = [self.chunks[i] for i in candidate_indices]
        cross_inputs = [[query, item["text"]] for item in candidate_chunks]
        cross_scores = self.cross_encoder.predict(cross_inputs, batch_size=min(16, len(cross_inputs)))

        ranked = sorted(zip(candidate_chunks, cross_scores), key=lambda pair: float(pair[1]), reverse=True)

        final_items: List[Dict[str, Any]] = []
        for rank, (chunk, score) in enumerate(ranked[:final_top_k], start=1):
            final_items.append({
                "rank": rank,
                "text": chunk["text"],
                "metadata": chunk["metadata"],
                "reranker_score": float(score),
            })
        return final_items

    @staticmethod
    def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
        blocks = []
        for item in retrieved:
            case_id = item["metadata"]["case_id"]
            blocks.append(f"[Case {case_id} | rank={item['rank']}] {item['text']}")
        return "\n\n---\n\n".join(blocks)

    def answer_query(
        self,
        query: str,
        llm: QwenAPIPipeline,
        retrieved: Optional[List[Dict[str, Any]]] = None,
    ) -> Tuple[str, List[Dict[str, Any]]]:
        retrieved = retrieved or self.search(query)
        if not retrieved:
            return "Я не знаю", []
        context = self.build_context_block(retrieved)
        system_prompt = (
            "Ты опытный врач-цитолог. Отвечай только на основе предоставленного контекста. "
            "Сначала назови диагностическую категорию Bethesda, затем дай одно короткое обоснование."
        )
        user_prompt = f"Контекст:\n{context}\n\nВопрос: {query}\nОтвет:"
        return llm.get_response(user_prompt, system_prompt), retrieved

    def cleanup(self) -> None:
        for attr in ["embedding_model", "cross_encoder", "bm25", "index", "chunks"]:
            if hasattr(self, attr):
                setattr(self, attr, None)
        clear_torch_memory()


In [5]:
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
from ragas.run_config import RunConfig

class AdvancedRAGEvaluator:
    def __init__(self, llm: Optional[QwenAPIPipeline] = None):
        self.llm = llm

    @staticmethod
    def parse_binary_judge(text: str) -> float:
        match = re.search(r"\b([01])\b", text)
        if match:
            return float(match.group(1))
        return np.nan

    def judge_binary(self, prompt: str) -> float:
        if self.llm is None:
            return np.nan
        raw = self.llm.get_response(prompt, system_prompt="Верни только 1 или 0.")
        return self.parse_binary_judge(raw)

    def generate_predictions(self, retrieval_rows: List[Dict[str, Any]]) -> pd.DataFrame:
        if self.llm is None:
            raise RuntimeError("Для генерации ответов нужен Qwen API.")
        records: List[Dict[str, Any]] = []
        for row in retrieval_rows:
            contexts = row["retrieved_contexts"]
            joined_context = "\n\n---\n\n".join(contexts)
            question = row["retrieval_query"]
            system_prompt = (
                "Ты опытный врач-цитолог. Используй только контекст. "
                "Верни категорию Bethesda и краткое обоснование."
            )
            user_prompt = f"Контекст:\n{joined_context}\n\nВопрос: {question}\nОтвет:"
            answer = self.llm.get_response(user_prompt, system_prompt=system_prompt)
            records.append({**row, "question": question, "answer": answer})
        return pd.DataFrame(records)

    def evaluate_manual(self, prediction_df: pd.DataFrame, label_key: str = "model") -> Tuple[Dict[str, Any], pd.DataFrame]:
        judged_df = prediction_df.copy()
        judged_df["hit_rate"] = judged_df["gold_case_hit"].astype(float)

        label_value = judged_df[label_key].iloc[0]
        if self.llm is None:
            summary = {
                label_key: label_value,
                "hit_rate": round(judged_df["hit_rate"].mean() * 100, 2),
                "context_relevance": np.nan,
                "faithfulness": np.nan,
                "answer_correctness": np.nan,
            }
            return summary, judged_df

        context_relevance_scores = []
        faithfulness_scores = []
        correctness_scores = []

        for _, row in judged_df.iterrows():
            context_text = "\n\n---\n\n".join(row["retrieved_contexts"])

            rel_prompt = (
                f"Контекст:\n{context_text}\n\n"
                f"Вопрос: {row['question']}\n"
                "Содержит ли найденный контекст достаточно информации, чтобы корректно определить категорию Bethesda?"
            )
            context_relevance_scores.append(self.judge_binary(rel_prompt))

            faith_prompt = (
                f"Контекст:\n{context_text}\n\n"
                f"Ответ: {row['answer']}\n"
                "Строго ли ответ основан на контексте без новых фактов?"
            )
            faithfulness_scores.append(self.judge_binary(faith_prompt))

            corr_prompt = (
                f"Вопрос: {row['question']}\n"
                f"Эталон: {row['ground_truth']}\n"
                f"Генерация: {row['answer']}\n"
                "Фактически корректен ли ответ относительно эталона?"
            )
            correctness_scores.append(self.judge_binary(corr_prompt))

        judged_df["context_relevance"] = context_relevance_scores
        judged_df["faithfulness_manual"] = faithfulness_scores
        judged_df["answer_correctness"] = correctness_scores

        summary = {
            label_key: label_value,
            "hit_rate": round(judged_df["hit_rate"].mean() * 100, 2),
            "context_relevance": round(judged_df["context_relevance"].mean() * 100, 2),
            "faithfulness": round(judged_df["faithfulness_manual"].mean() * 100, 2),
            "answer_correctness": round(judged_df["answer_correctness"].mean() * 100, 2),
        }
        return summary, judged_df

def safe_metric_mean(frame: pd.DataFrame, column: str) -> float:
    if column not in frame.columns:
        return np.nan
    series = pd.to_numeric(frame[column], errors="coerce")
    return round(series.mean() * 100, 2) if series.notna().any() else np.nan

def coerce_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return str(value).strip()

def normalize_ragas_contexts(value: Any) -> List[str]:
    if isinstance(value, list):
        return [str(item).strip() for item in value if item is not None and not (isinstance(item, float) and np.isnan(item))]
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    return [str(value).strip()]

def prediction_frame_to_ragas_dataset(prediction_df: pd.DataFrame) -> EvaluationDataset:
    records: List[Dict[str, Any]] = []
    for row in prediction_df.to_dict(orient="records"):
        records.append({
            "user_input": coerce_text(row.get("question") or row.get("retrieval_query")),
            "response": coerce_text(row.get("answer")),
            "retrieved_contexts": normalize_ragas_contexts(row.get("retrieved_contexts")),
            "reference": coerce_text(row.get("ground_truth")),
        })
    return EvaluationDataset.from_list(records)

def evaluate_with_ragas(model_label: str, model_name: str, prediction_df: pd.DataFrame, llm: QwenAPIPipeline) -> Tuple[Dict[str, Any], pd.DataFrame]:
    ragas_llm = llm.get_chat_model()
    ragas_embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    dataset = prediction_frame_to_ragas_dataset(prediction_df)
    result = evaluate(
        dataset=dataset,
        metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=RunConfig(max_workers=1, timeout=180),
        raise_exceptions=False,
    )
    df = result.to_pandas()
    summary = {
        "model": model_label,
        "context_precision": safe_metric_mean(df, "context_precision"),
        "context_recall": safe_metric_mean(df, "context_recall"),
        "faithfulness_ragas": safe_metric_mean(df, "faithfulness"),
        "answer_relevancy": safe_metric_mean(df, "answer_relevancy"),
    }
    return summary, df


In [6]:
retrieval_summary_df = pd.DataFrame()
retrieval_payload: Dict[str, Any] = {}
manual_summary_df = pd.DataFrame()
prediction_frames: Dict[str, pd.DataFrame] = {}
manual_detail_frames: Dict[str, pd.DataFrame] = {}
ragas_summary_df = pd.DataFrame()
ragas_detail_frames: Dict[str, pd.DataFrame] = {}

def run_retrieval_sweep(model_specs: List[ModelSpec]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    summary_rows: List[Dict[str, Any]] = []
    payload: Dict[str, Any] = {}

    for spec in model_specs:
        print("=" * 80)
        print(f"Running retrieval sweep for {spec.label} -> {spec.model_name}")
        pipeline = None
        try:
            clear_torch_memory()
            pipeline = AdvancedRAGPipeline(spec)
            pipeline.process_documents(documents)

            model_rows: List[Dict[str, Any]] = []
            reciprocal_ranks: List[float] = []

            for row in eval_rows:
                retrieved = pipeline.search(row["retrieval_query"])
                retrieved_ids = [item["metadata"]["case_id"] for item in retrieved]
                retrieved_contexts = [item["text"] for item in retrieved]
                rank = next((idx + 1 for idx, case_id in enumerate(retrieved_ids) if case_id == row["id"]), None)
                reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)

                model_rows.append({
                    "model": spec.label,
                    "hf_model": spec.model_name,
                    "query_id": row["id"],
                    "question": row["retrieval_query"],
                    "retrieval_query": row["retrieval_query"],
                    "case_anchor": row["case_anchor"],
                    "ground_truth": row["ground_truth"],
                    "bethesda_label": row["bethesda_label"],
                    "gold_case_id": row["id"],
                    "retrieved_case_ids": retrieved_ids,
                    "retrieved_contexts": retrieved_contexts,
                    "gold_case_hit": int(row["id"] in retrieved_ids),
                    "gold_case_rank": rank,
                })

            summary_rows.append({
                "model": spec.label,
                "hf_model": spec.model_name,
                "status": "ok",
                "hit_rate": round(np.mean([row["gold_case_hit"] for row in model_rows]) * 100, 2),
                "mrr": round(np.mean(reciprocal_ranks), 4),
                "batch_size": spec.batch_size,
                "max_length": spec.max_length,
                "notes": spec.notes,
            })
            payload[spec.label] = {"status": "ok", "spec": asdict(spec), "rows": model_rows}
        except Exception as exc:
            summary_rows.append({
                "model": spec.label,
                "hf_model": spec.model_name,
                "status": "failed",
                "hit_rate": np.nan,
                "mrr": np.nan,
                "batch_size": spec.batch_size,
                "max_length": spec.max_length,
                "notes": f"{spec.notes} | error={type(exc).__name__}: {exc}",
            })
            payload[spec.label] = {"status": "failed", "spec": asdict(spec), "error": f"{type(exc).__name__}: {exc}", "rows": []}
        finally:
            if pipeline is not None:
                pipeline.cleanup()

    summary_df = pd.DataFrame(summary_rows).sort_values(["status", "hit_rate"], ascending=[True, False])
    return summary_df, payload

def build_prediction_frame_if_needed(model_label: str, rows: List[Dict[str, Any]], llm: QwenAPIPipeline) -> pd.DataFrame:
    if model_label in prediction_frames:
        return prediction_frames[model_label]
    evaluator = AdvancedRAGEvaluator(llm=llm)
    prediction_df = evaluator.generate_predictions(rows)
    prediction_frames[model_label] = prediction_df
    return prediction_df

if RUN_RETRIEVAL_SWEEP:
    retrieval_summary_df, retrieval_payload = run_retrieval_sweep(MODEL_SPECS)
    retrieval_summary_df.to_csv(EMBEDDING_ARTIFACT_DIR / "retrieval_summary.csv", index=False, encoding="utf-8-sig")
    (EMBEDDING_ARTIFACT_DIR / "retrieval_payload.json").write_text(json.dumps(retrieval_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    display(retrieval_summary_df)
else:
    payload_path = EMBEDDING_ARTIFACT_DIR / "retrieval_payload.json"
    summary_path = EMBEDDING_ARTIFACT_DIR / "retrieval_summary.csv"
    if payload_path.exists() and summary_path.exists():
        retrieval_payload = json.loads(payload_path.read_text(encoding="utf-8"))
        retrieval_summary_df = pd.read_csv(summary_path)
        display(retrieval_summary_df)
    else:
        print("Retrieval sweep skipped and no cached payload found.")

if RUN_LLM_MANUAL_METRICS:
    evaluator = AdvancedRAGEvaluator(llm=get_qwen_api())
    manual_rows: List[Dict[str, Any]] = []
    for spec in MODEL_SPECS:
        payload = retrieval_payload.get(spec.label, {})
        if payload.get("status") != "ok":
            continue
        prediction_df = evaluator.generate_predictions(payload["rows"])
        summary, judged_df = evaluator.evaluate_manual(prediction_df, label_key="model")
        prediction_frames[spec.label] = prediction_df
        manual_detail_frames[spec.label] = judged_df
        manual_rows.append(summary)
    manual_summary_df = pd.DataFrame(manual_rows)
    if not manual_summary_df.empty:
        manual_summary_df = manual_summary_df.sort_values("hit_rate", ascending=False)
        manual_summary_df.to_csv(EMBEDDING_ARTIFACT_DIR / "manual_metrics_summary.csv", index=False, encoding="utf-8-sig")
        display(manual_summary_df)
        pd.concat(manual_detail_frames.values(), ignore_index=True).to_json(
            EMBEDDING_ARTIFACT_DIR / "manual_metrics_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
else:
    print("RUN_LLM_MANUAL_METRICS=False -> manual Qwen-judge metrics skipped.")

if RUN_RAGAS:
    ragas_rows: List[Dict[str, Any]] = []
    for spec in MODEL_SPECS:
        payload = retrieval_payload.get(spec.label, {})
        if payload.get("status") != "ok":
            continue
        prediction_df = build_prediction_frame_if_needed(spec.label, payload["rows"], get_qwen_api())
        summary, detail_df = evaluate_with_ragas(spec.label, spec.model_name, prediction_df, get_qwen_api())
        ragas_rows.append(summary)
        ragas_detail_frames[spec.label] = detail_df
    ragas_summary_df = pd.DataFrame(ragas_rows)
    if not ragas_summary_df.empty:
        ragas_summary_df = ragas_summary_df.sort_values("context_precision", ascending=False)
        ragas_summary_df.to_csv(EMBEDDING_ARTIFACT_DIR / "ragas_metrics_summary.csv", index=False, encoding="utf-8-sig")
        pd.concat(ragas_detail_frames, names=["model", "row_id"]).reset_index().to_json(
            EMBEDDING_ARTIFACT_DIR / "ragas_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
        display(ragas_summary_df)
else:
    print("RUN_RAGAS=False -> ragas evaluation skipped.")

comparison_df = retrieval_summary_df[["model", "hf_model", "hit_rate", "mrr", "status"]].copy() if not retrieval_summary_df.empty else pd.DataFrame()
if not comparison_df.empty:
    comparison_df = comparison_df.rename(columns={"hit_rate": "hit_rate_retrieval"})
    if not manual_summary_df.empty:
        comparison_df = comparison_df.merge(manual_summary_df, on="model", how="left")
    if not ragas_summary_df.empty:
        comparison_df = comparison_df.merge(ragas_summary_df, on="model", how="left")
    comparison_path = EMBEDDING_ARTIFACT_DIR / "combined_metrics.csv"
    comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
    display(comparison_df)
    print(f"Combined metrics saved to: {comparison_path}")
    if not existing_ragas_baseline_df.empty:
        print("Existing project-level ragas baseline from ragas_evaluation_results.csv")
        display(existing_ragas_baseline_df)


Retrieval sweep skipped and no cached payload found.
RUN_LLM_MANUAL_METRICS=False -> manual Qwen-judge metrics skipped.
RUN_RAGAS=False -> ragas evaluation skipped.


## Chunking Strategy Study

- Reuse the same retrieval and evaluation stack, but keep the embedding backbone fixed.
- Compare realistic chunking strategies for short Russian cytology reports.
- Keep Qwen as the only external LLM when answer generation or judge metrics are enabled.


In [7]:
@dataclass
class ChunkingSpec:
    label: str
    notes: str

CHUNKING_STRATEGIES: List[ChunkingSpec] = [
    ChunkingSpec("whole_case", "Baseline: one cytology case per indexed document"),
    ChunkingSpec("sentence_window_2", "Sliding 2-sentence windows with overlap=1"),
    ChunkingSpec("structural_subcase", "Split by lobe or subcase markers when present"),
    ChunkingSpec("structural_then_sentence_window_2", "Split by subcase first, then 2-sentence windows inside each segment"),
]

CHUNKING_BACKBONE_LABEL = os.getenv("CHUNKING_BACKBONE_LABEL", "rubert_tiny2_baseline")

def resolve_model_spec(label: str, specs: List[ModelSpec]) -> ModelSpec:
    for spec in specs:
        if spec.label == label:
            return spec
    available = ", ".join(spec.label for spec in specs)
    raise ValueError(f"Unknown model label: {label}. Available: {available}")

CHUNKING_MODEL_SPEC = resolve_model_spec(CHUNKING_BACKBONE_LABEL, MODEL_SPECS)

def split_sentences_ru(text: str) -> List[str]:
    sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]
    return sentences or [text.strip()]

def is_subcase_header(sentence: str) -> bool:
    header_markers = ("пр. доля", "лев. доля", "п ", "л ", "п/п", "н/3", "прав.", "лев.")
    sentence_lc = sentence.lower()
    if "№" not in sentence and not any(sentence_lc.startswith(marker) for marker in header_markers):
        return False
    if len(sentence) > 90 and sentence.count(".") < 2:
        return False
    return any(sentence_lc.startswith(marker) for marker in header_markers) or "№" in sentence

def split_structural_subcases(text: str) -> List[str]:
    sentences = split_sentences_ru(text)
    segments: List[List[str]] = []
    current: List[str] = []
    for sentence in sentences:
        if is_subcase_header(sentence) and current:
            segments.append(current)
            current = [sentence]
        else:
            current.append(sentence)
    if current:
        segments.append(current)
    return [" ".join(segment).strip() for segment in segments if " ".join(segment).strip()]

def build_sentence_windows(sentences: List[str], window_size: int = 2, overlap: int = 1) -> List[str]:
    if len(sentences) <= window_size:
        return [" ".join(sentences).strip()]
    step = max(1, window_size - overlap)
    windows: List[str] = []
    for start in range(0, len(sentences), step):
        window = sentences[start : start + window_size]
        if not window:
            continue
        windows.append(" ".join(window).strip())
        if start + window_size >= len(sentences):
            break
    return windows

def make_chunk_record(text: str, row: Dict[str, Any], chunking_label: str, chunk_index: int, parent_segment_index: int = 0) -> Tuple[str, Dict[str, Any]]:
    return (
        text,
        {
            "case_id": row["id"],
            "case_anchor": row["case_anchor"],
            "chunking": chunking_label,
            "chunk_id": f"{row['id']}::{chunking_label}::{chunk_index}",
            "parent_segment_index": parent_segment_index,
        },
    )

def build_chunked_documents(rows: List[Dict[str, Any]], spec: ChunkingSpec) -> List[Tuple[str, Dict[str, Any]]]:
    documents: List[Tuple[str, Dict[str, Any]]] = []
    for row in rows:
        text = row["simulated_context"]
        if spec.label == "whole_case":
            documents.append(make_chunk_record(text, row, spec.label, chunk_index=0))
            continue
        if spec.label == "sentence_window_2":
            windows = build_sentence_windows(split_sentences_ru(text), window_size=2, overlap=1)
            for idx, window in enumerate(windows):
                documents.append(make_chunk_record(window, row, spec.label, chunk_index=idx))
            continue
        if spec.label == "structural_subcase":
            segments = split_structural_subcases(text)
            for idx, segment in enumerate(segments):
                documents.append(make_chunk_record(segment, row, spec.label, chunk_index=idx, parent_segment_index=idx))
            continue
        if spec.label == "structural_then_sentence_window_2":
            segments = split_structural_subcases(text)
            chunk_index = 0
            for segment_idx, segment in enumerate(segments):
                windows = build_sentence_windows(split_sentences_ru(segment), window_size=2, overlap=1)
                for window in windows:
                    documents.append(make_chunk_record(window, row, spec.label, chunk_index=chunk_index, parent_segment_index=segment_idx))
                    chunk_index += 1
            continue
        raise ValueError(f"Unsupported chunking strategy: {spec.label}")
    return documents

chunking_preview_rows: List[Dict[str, Any]] = []
for spec in CHUNKING_STRATEGIES:
    docs = build_chunked_documents(eval_rows, spec)
    chunking_preview_rows.append({
        "chunking": spec.label,
        "chunks_total": len(docs),
        "avg_chunks_per_case": round(len(docs) / len(eval_rows), 2),
        "notes": spec.notes,
    })
chunking_preview_df = pd.DataFrame(chunking_preview_rows)
display(chunking_preview_df)
print({
    "chunking_backbone_label": CHUNKING_MODEL_SPEC.label,
    "chunking_backbone_model": CHUNKING_MODEL_SPEC.model_name,
    "chunking_artifacts": str(CHUNKING_ARTIFACT_DIR),
    "chunking_backbone_note": "Default chunking backbone is the lightest embedding model to avoid long CPU startup.",
})


,chunking,chunks_total,avg_chunks_per_case,notes
0,whole_case,10,1.0,Baseline: one cytology case per indexed document
1,sentence_window_2,21,2.1,Sliding 2-sentence windows with overlap=1
2,structural_subcase,15,1.5,Split by lobe or subcase markers when present
3,structural_then_sentence_window_2,18,1.8,"Split by subcase first, then 2-sentence window..."


{'chunking_backbone_label': 'rubert_tiny2_baseline', 'chunking_backbone_model': 'cointegrated/rubert-tiny2', 'chunking_artifacts': 'd:\\Mephi\\22kaf\\6sem\\УИР\\CytoRAG\\artifacts\\qwen_api_embedding_chunking_study\\chunking_study', 'chunking_backbone_note': 'Default chunking backbone is the lightest embedding model to avoid long CPU startup.'}


In [8]:
chunking_retrieval_summary_df = pd.DataFrame()
chunking_retrieval_payload: Dict[str, Any] = {}
chunking_prediction_frames: Dict[str, pd.DataFrame] = {}
chunking_ragas_summary_df = pd.DataFrame()
chunking_ragas_detail_frames: Dict[str, pd.DataFrame] = {}
chunking_manual_summary_df = pd.DataFrame()
chunking_manual_detail_frames: Dict[str, pd.DataFrame] = {}

def run_chunking_sweep(chunking_specs: List[ChunkingSpec], embedding_spec: ModelSpec) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    summary_rows: List[Dict[str, Any]] = []
    payload: Dict[str, Any] = {}

    for chunk_spec in chunking_specs:
        print("=" * 80)
        print(f"Running chunking sweep for {chunk_spec.label} using {embedding_spec.label} -> {embedding_spec.model_name}")
        pipeline = None
        chunked_documents = build_chunked_documents(eval_rows, chunk_spec)
        try:
            clear_torch_memory()
            pipeline = AdvancedRAGPipeline(embedding_spec)
            pipeline.process_documents(chunked_documents)

            model_rows: List[Dict[str, Any]] = []
            reciprocal_ranks: List[float] = []

            for row in eval_rows:
                retrieved = pipeline.search(row["retrieval_query"])
                retrieved_case_ids = [item["metadata"]["case_id"] for item in retrieved]
                retrieved_chunk_ids = [item["metadata"]["chunk_id"] for item in retrieved]
                retrieved_contexts = [item["text"] for item in retrieved]
                rank = next((idx + 1 for idx, item in enumerate(retrieved) if item["metadata"]["case_id"] == row["id"]), None)
                reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)

                model_rows.append({
                    "chunking": chunk_spec.label,
                    "model": chunk_spec.label,
                    "hf_model": embedding_spec.model_name,
                    "query_id": row["id"],
                    "question": row["retrieval_query"],
                    "retrieval_query": row["retrieval_query"],
                    "case_anchor": row["case_anchor"],
                    "ground_truth": row["ground_truth"],
                    "bethesda_label": row["bethesda_label"],
                    "gold_case_id": row["id"],
                    "retrieved_case_ids": retrieved_case_ids,
                    "retrieved_chunk_ids": retrieved_chunk_ids,
                    "retrieved_contexts": retrieved_contexts,
                    "gold_case_hit": int(row["id"] in retrieved_case_ids),
                    "gold_case_rank": rank,
                })

            summary_rows.append({
                "chunking": chunk_spec.label,
                "hf_model": embedding_spec.model_name,
                "status": "ok",
                "hit_rate": round(np.mean([row["gold_case_hit"] for row in model_rows]) * 100, 2),
                "mrr": round(np.mean(reciprocal_ranks), 4),
                "indexed_chunks": len(chunked_documents),
                "avg_chunks_per_case": round(len(chunked_documents) / len(eval_rows), 2),
                "notes": chunk_spec.notes,
            })
            payload[chunk_spec.label] = {
                "status": "ok",
                "spec": asdict(chunk_spec),
                "embedding_spec": asdict(embedding_spec),
                "indexed_chunks": len(chunked_documents),
                "rows": model_rows,
            }
        except Exception as exc:
            summary_rows.append({
                "chunking": chunk_spec.label,
                "hf_model": embedding_spec.model_name,
                "status": "failed",
                "hit_rate": np.nan,
                "mrr": np.nan,
                "indexed_chunks": len(chunked_documents),
                "avg_chunks_per_case": round(len(chunked_documents) / len(eval_rows), 2),
                "notes": f"{chunk_spec.notes} | error={type(exc).__name__}: {exc}",
            })
            payload[chunk_spec.label] = {
                "status": "failed",
                "spec": asdict(chunk_spec),
                "embedding_spec": asdict(embedding_spec),
                "indexed_chunks": len(chunked_documents),
                "error": f"{type(exc).__name__}: {exc}",
                "rows": [],
            }
        finally:
            if pipeline is not None:
                pipeline.cleanup()

    summary_df = pd.DataFrame(summary_rows).sort_values(["status", "hit_rate"], ascending=[True, False])
    return summary_df, payload

def build_chunking_prediction_frame_if_needed(chunking_label: str, rows: List[Dict[str, Any]], llm: QwenAPIPipeline) -> pd.DataFrame:
    if chunking_label in chunking_prediction_frames:
        return chunking_prediction_frames[chunking_label]
    evaluator = AdvancedRAGEvaluator(llm=llm)
    prediction_df = evaluator.generate_predictions(rows)
    chunking_prediction_frames[chunking_label] = prediction_df
    return prediction_df

if RUN_CHUNKING_SWEEP:
    chunking_retrieval_summary_df, chunking_retrieval_payload = run_chunking_sweep(CHUNKING_STRATEGIES, CHUNKING_MODEL_SPEC)
    chunking_retrieval_summary_df.to_csv(CHUNKING_ARTIFACT_DIR / "chunking_retrieval_summary.csv", index=False, encoding="utf-8-sig")
    (CHUNKING_ARTIFACT_DIR / "chunking_retrieval_payload.json").write_text(json.dumps(chunking_retrieval_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    display(chunking_retrieval_summary_df)
else:
    payload_path = CHUNKING_ARTIFACT_DIR / "chunking_retrieval_payload.json"
    summary_path = CHUNKING_ARTIFACT_DIR / "chunking_retrieval_summary.csv"
    if payload_path.exists() and summary_path.exists():
        chunking_retrieval_payload = json.loads(payload_path.read_text(encoding="utf-8"))
        chunking_retrieval_summary_df = pd.read_csv(summary_path)
        display(chunking_retrieval_summary_df)
    else:
        print("Chunking sweep skipped and no cached payload found.")

if RUN_LLM_MANUAL_METRICS:
    evaluator = AdvancedRAGEvaluator(llm=get_qwen_api())
    manual_rows: List[Dict[str, Any]] = []
    for chunk_spec in CHUNKING_STRATEGIES:
        payload = chunking_retrieval_payload.get(chunk_spec.label, {})
        if payload.get("status") != "ok":
            continue
        prediction_df = build_chunking_prediction_frame_if_needed(chunk_spec.label, payload["rows"], get_qwen_api())
        summary, judged_df = evaluator.evaluate_manual(prediction_df, label_key="chunking")
        chunking_manual_detail_frames[chunk_spec.label] = judged_df
        manual_rows.append(summary)
    chunking_manual_summary_df = pd.DataFrame(manual_rows)
    if not chunking_manual_summary_df.empty:
        chunking_manual_summary_df = chunking_manual_summary_df.sort_values("hit_rate", ascending=False)
        chunking_manual_summary_df.to_csv(CHUNKING_ARTIFACT_DIR / "chunking_manual_metrics_summary.csv", index=False, encoding="utf-8-sig")
        pd.concat(chunking_manual_detail_frames.values(), ignore_index=True).to_json(
            CHUNKING_ARTIFACT_DIR / "chunking_manual_metrics_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
        display(chunking_manual_summary_df)

if RUN_RAGAS:
    ragas_rows: List[Dict[str, Any]] = []
    for chunk_spec in CHUNKING_STRATEGIES:
        payload = chunking_retrieval_payload.get(chunk_spec.label, {})
        if payload.get("status") != "ok":
            continue
        prediction_df = build_chunking_prediction_frame_if_needed(chunk_spec.label, payload["rows"], get_qwen_api())
        summary, detail_df = evaluate_with_ragas(chunk_spec.label, CHUNKING_MODEL_SPEC.model_name, prediction_df, get_qwen_api())
        summary["chunking"] = summary.pop("model")
        summary["hf_model"] = CHUNKING_MODEL_SPEC.model_name
        summary["notes"] = chunk_spec.notes
        ragas_rows.append(summary)
        chunking_ragas_detail_frames[chunk_spec.label] = detail_df
    chunking_ragas_summary_df = pd.DataFrame(ragas_rows)
    if not chunking_ragas_summary_df.empty:
        chunking_ragas_summary_df = chunking_ragas_summary_df.sort_values("context_precision", ascending=False)
        chunking_ragas_summary_df.to_csv(CHUNKING_ARTIFACT_DIR / "chunking_ragas_metrics_summary.csv", index=False, encoding="utf-8-sig")
        pd.concat(chunking_ragas_detail_frames, names=["chunking", "row_id"]).reset_index().to_json(
            CHUNKING_ARTIFACT_DIR / "chunking_ragas_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
        display(chunking_ragas_summary_df)

chunking_comparison_df = chunking_retrieval_summary_df[["chunking", "hf_model", "hit_rate", "mrr", "indexed_chunks", "avg_chunks_per_case", "status", "notes"]].copy() if not chunking_retrieval_summary_df.empty else pd.DataFrame()
if not chunking_comparison_df.empty:
    chunking_comparison_df = chunking_comparison_df.rename(columns={"hit_rate": "hit_rate_retrieval"})
    if not chunking_manual_summary_df.empty:
        chunking_comparison_df = chunking_comparison_df.merge(chunking_manual_summary_df, on="chunking", how="left")
    if not chunking_ragas_summary_df.empty:
        chunking_comparison_df = chunking_comparison_df.merge(
            chunking_ragas_summary_df[["chunking", "context_precision", "context_recall", "faithfulness_ragas", "answer_relevancy"]],
            on="chunking",
            how="left",
        )
    chunking_comparison_path = CHUNKING_ARTIFACT_DIR / "chunking_combined_metrics.csv"
    chunking_comparison_df.to_csv(chunking_comparison_path, index=False, encoding="utf-8-sig")

    ranking_columns = [column for column in ["context_precision", "context_recall", "hit_rate_retrieval", "mrr"] if column in chunking_comparison_df.columns]
    ranked_chunking_df = chunking_comparison_df.sort_values(ranking_columns, ascending=[False] * len(ranking_columns), na_position="last") if ranking_columns else chunking_comparison_df.copy()
    chunking_report_table = dataframe_to_markdown(
        ranked_chunking_df,
        columns=[column for column in ["chunking", "hit_rate_retrieval", "mrr", "context_precision", "context_recall", "faithfulness_ragas", "answer_relevancy", "indexed_chunks"] if column in ranked_chunking_df.columns],
    )
    top_chunking_rows = ranked_chunking_df.head(2)
    observations: List[str] = []
    if not top_chunking_rows.empty:
        best_row = top_chunking_rows.iloc[0]
        observations.append(f"- Best chunking by primary ranking: `{best_row['chunking']}`.")
        observations.append(f"- Retrieval hit rate: `{best_row.get('hit_rate_retrieval', np.nan):.2f}`, indexed chunks: `{int(best_row.get('indexed_chunks', 0))}`.")
        if len(top_chunking_rows) > 1:
            runner_up = top_chunking_rows.iloc[1]
            observations.append(f"- Runner-up: `{runner_up['chunking']}`.")
    report_lines = [
        "# CytoRAG Chunking Study (Qwen API)",
        "",
        "## Configuration",
        "",
        f"- Backbone embedding label: `{CHUNKING_MODEL_SPEC.label}`",
        f"- Backbone embedding model: `{CHUNKING_MODEL_SPEC.model_name}`",
        f"- Qwen judge model: `{QWEN_MODEL_REPO}`",
        f"- Reranker: `{RERANKER_NAME}`",
        f"- Dataset: `{DATA_PATH.name}`",
        "",
        "## Combined Metrics",
        "",
        chunking_report_table,
        "",
        "## Key Observations",
        "",
        *(observations if observations else ["- No completed runs were available."]),
    ]
    chunking_report_path = CHUNKING_ARTIFACT_DIR / "chunking_report.md"
    chunking_report_path.write_text("\n".join(report_lines), encoding="utf-8")
    display(chunking_comparison_df)
    print(f"Chunking combined metrics saved to: {chunking_comparison_path}")
    print(f"Chunking markdown report saved to: {chunking_report_path}")


Running chunking sweep for whole_case using rubert_tiny2_baseline -> cointegrated/rubert-tiny2
Loading embedding model: cointegrated/rubert-tiny2 on cpu


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 4653.57it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker: DiTy/cross-encoder-russian-msmarco on cpu


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3905.31it/s]


Running chunking sweep for sentence_window_2 using rubert_tiny2_baseline -> cointegrated/rubert-tiny2
Loading embedding model: cointegrated/rubert-tiny2 on cpu


...

Loading reranker: DiTy/cross-encoder-russian-msmarco on cpu


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4212.75it/s]


,chunking,hf_model,status,hit_rate,mrr,indexed_chunks,avg_chunks_per_case,notes
0,whole_case,cointegrated/rubert-tiny2,failed,NaN,NaN,10,1.0,Baseline: one cytology case per indexed docume...
1,sentence_window_2,cointegrated/rubert-tiny2,failed,NaN,NaN,21,2.1,Sliding 2-sentence windows with overlap=1 | er...
2,structural_subcase,cointegrated/rubert-tiny2,failed,NaN,NaN,15,1.5,Split by lobe or subcase markers when present ...
3,structural_then_sentence_window_2,cointegrated/rubert-tiny2,failed,NaN,NaN,18,1.8,"Split by subcase first, then 2-sentence window..."


,chunking,hf_model,hit_rate_retrieval,mrr,indexed_chunks,avg_chunks_per_case,status,notes
0,whole_case,cointegrated/rubert-tiny2,NaN,NaN,10,1.0,failed,Baseline: one cytology case per indexed docume...
1,sentence_window_2,cointegrated/rubert-tiny2,NaN,NaN,21,2.1,failed,Sliding 2-sentence windows with overlap=1 | er...
2,structural_subcase,cointegrated/rubert-tiny2,NaN,NaN,15,1.5,failed,Split by lobe or subcase markers when present ...
3,structural_then_sentence_window_2,cointegrated/rubert-tiny2,NaN,NaN,18,1.8,failed,"Split by subcase first, then 2-sentence window..."


Chunking combined metrics saved to: d:\Mephi\22kaf\6sem\УИР\CytoRAG\artifacts\qwen_api_embedding_chunking_study\chunking_study\chunking_combined_metrics.csv
Chunking markdown report saved to: d:\Mephi\22kaf\6sem\УИР\CytoRAG\artifacts\qwen_api_embedding_chunking_study\chunking_study\chunking_report.md


## Next steps

- Для отдельного запуска чанкинга выставь `EXPERIMENT_MODE="chunking_only"`; по умолчанию он возьмёт лёгкий backbone `rubert_tiny2_baseline`.
- Если захочешь вернуть более тяжёлую модель, задай `CHUNKING_BACKBONE_LABEL` вручную, например `bge_m3`.
- Если нужен полный LLM-judge/RAGAS прогон, переключи `RUN_LLM_MANUAL_METRICS=True` и/или `RUN_RAGAS=True`.
- Если стоимость API важна, сначала прогони retrieval-only sweep, а затем включай Qwen только для top-кандидатов.
- При необходимости можно добавить отдельную smoke-test ячейку для Qwen API перед полным запуском.
